# Data Preprocessing : Scaling, Encoding, Normalization ( with scikit-learn)

## Concept

### What is data Preprocessing ?

**Data Preprocessing** transform raw data into clean and usable format. It imporves model performance and training speed.

### Common Tasks:

- Handling missing values
- Scaling numerical features
- Encoding categorical variables
- Normalizing feature ranges

### 1. Feature Scaling

- Scaling adjust values so feature contribute equally.
- Common Techniques: StandardScaler(Z-Score), MinMaxScaler(0-1 scale)

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
# Example Dataset
numeric_data = pd.DataFrame({
    'Salary':[30000, 45000, 60000, 80000, 120000],
    'Age': [25, 32, 47, 51, 62]
})

numeric_data

,Salary,Age
0,30000,25
1,45000,32
2,60000,47
3,80000,51
4,120000,62


**Quiz: Which feature (column) from this dataframe, machine learning model will give more important? and why?**

**Answer: The model will give more importance to Salary because it has a larger scale compared to Age. Many ML algorithms are sensitive to feature magnitude, so without scaling, Salary dominates the model. This can be fixed using feature scaling techniques.**

In [2]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
# StandardScalaer(Z-score)
std_scaler = StandardScaler()
scaled_std = std_scaler.fit_transform(numeric_data)
df = pd.DataFrame(scaled_std, columns = numeric_data.columns)
print("\nStandard Scalaed Data:\n", df)


Standard Scalaed Data:
      Salary       Age
0 -1.184341 -1.382872
1 -0.704203 -0.856780
2 -0.224065  0.270562
3  0.416120  0.571186
4  1.696489  1.397904


### fit_transform() does two step at once:

- **fit():** Learns the mean & Standard Deviation of your data.
- **transform():** Applies the scaling (converts data to Z-scores)
- For `StandardScaler`: Each feature becomes mean = 0 and std = 1)

### How many Standard Deviation away each value is from the mean.

### Example:

**If an employee's age is 75 in a class where:**

**Mean = 70**

**Standard Deviation = 5**

**Z-Score = (75-70)/5 = +1.0 (1 Standard Deviation above average)**

### MinMaxScaler(0-1)

In [3]:
minmax_scaler = MinMaxScaler()
scaled_minmax = minmax_scaler.fit_transform(numeric_data)
print("\nMin-Max Scaled Data:\n", pd.DataFrame(scaled_minmax, columns = numeric_data.columns))


Min-Max Scaled Data:
      Salary       Age
0  0.000000  0.000000
1  0.166667  0.189189
2  0.333333  0.594595
3  0.555556  0.702703
4  1.000000  1.000000


**Xscaled = (X -Xmin)/(Xmax-Xmin)**

**1st Value of age : 25-25/(62-25) = 0**

**2nd Value of age : (32-25)/62-25) = 0.18**

## StandardScaler Vs. MinMaxScaler Ranges

|**Scaler** | **Typical Range** | **What It Means** |
|-------|---------------|----------------|
|**MinMaxScaler**| `[0,1]`    | "Squeezed data into a fixed 0-1 box."|
|**StandardScaler**| **No fixed range!** | "Centers data around 0; most values fall between `-3` to `+3` (99% of data)."|

### 2. Encoding Categorical Variables

#### Convert text labels to numeric format for ML models

In [4]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# Sample Data

cat_data = pd.DataFrame({
    'Department': ['HR','Sales','IT','HR','IT']
})
cat_data

,Department
0,HR
1,Sales
2,IT
3,HR
4,IT


### Label Encoding

In [5]:
label_encoder = LabelEncoder()
cat_data['Dept_Label'] = label_encoder.fit_transform(cat_data['Department'])
print("\nLabel Encoded:\n", cat_data)


Label Encoded:
   Department  Dept_Label
0         HR           0
1      Sales           2
2         IT           1
3         HR           0
4         IT           1


### fit_transform() does two step at once:

- **fit():** Learns all unique categories in the column
- **transform():** Converts each category to a number (e.g., HR-> 0, IT ->1)


### One_Hot Encoding - Alternative to label encoder

In [7]:
cat_data_ohe = pd.get_dummies(cat_data['Department'], prefix = 'Dept').astype(int)
print("\nOne-Hot Encoded:\n\n",cat_data_ohe)


One-Hot Encoded:

    Dept_HR  Dept_IT  Dept_Sales
0        1        0           0
1        0        0           1
2        0        1           0
3        1        0           0
4        0        1           0


## Label Encoder Vs. One-Hot Encoding

### label Encoder

- Output:`['HR',IT'] -> [0,1]`
- Use for **Target(y)** only
- Never use for : Features(creates false order)

### One-Hot(get_dummies)

- Output:`'HR' ->[1,0], 'IT' -> [0,1]`
- Use for **Features(x)** only
- Never use for : > 15 categories


### 3. Normalization

**Normalize data such that row vector magnitudes = 1**

#### Why use Normalizer?

##### Purpose:

Scales each row to have a unit maginitude(length = 1) while preserving direction.

##### Key Idea:

Converts row vectors into "pure direction" form by dividing by their Euclidean norm (√(x₁² + x₂² + ...)).

##### When to Use:

- Text data(TF-IDF, word counts)
- Clustering(k-means, cosine similarity)

Any algorithm sensitive to vector magnitudes but not their lengts(e.g., NLP, recommender systems).


In [8]:
from sklearn.preprocessing import Normalizer

# Example Data

X = np.array([[3.0,4.0], [1.0,0.0],[0.0,8.0]])
print("\nOriginal Data:\n",X)
normalizer = Normalizer()
normalized_data = normalizer.fit_transform(X)
print("\nNormalized Data:\n", normalized_data)


Original Data:
 [[3. 4.]
 [1. 0.]
 [0. 8.]]

Normalized Data:
 [[0.6 0.8]
 [1.  0. ]
 [0.  1. ]]


### How It Works

For each row, the Normalizer: Computes the Euclidean norm (magnitude):

norm = sqrt(x₁² + x₂² + ...)

Divides each element by the norm:

x₁_normalized = x₁ / norm, x₂_normalized = x₂ / norm

Calculates Row Norm: For [3, 4] → norm = √(3² + 4²) = 5

Divides Each Element by Norm: [3/5, 4/5] = [0.6, 0.8] (now has length 1).


### Why Not MinMax/StandardScaler?

**MinMax/StandardScaler compares features across rows(columns), but Normalizer scales each row(sample) to focus on relative feature productions.**

# Case Study: Retail Customer Data

## Scenario:

You have a retail customer dataset with income, age and gender.

You want to prepare it for ML model input.

In [9]:
raw_data = pd.DataFrame({
    'Customer_ID' : [101, 102, 103, 104, 105],
    'Gender': ['Male', 'Female', 'Female', 'Male', 'Female'],
    'Age': [23, 45, 31, 35, 52],
    'income': [25000, 52000, 38000, 45000, 61000]
})

raw_data

,Customer_ID,Gender,Age,income
0,101,Male,23,25000
1,102,Female,45,52000
2,103,Female,31,38000
3,104,Male,35,45000
4,105,Female,52,61000


In [10]:
# Drop ID (not useful for training)
data = raw_data.drop(columns = 'Customer_ID')
data

,Gender,Age,income
0,Male,23,25000
1,Female,45,52000
2,Female,31,38000
3,Male,35,45000
4,Female,52,61000


In [13]:
data['Gender'] = label_encoder.fit_transform(data['Gender'])
data

,Gender,Age,income
0,1,23,25000
1,0,45,52000
2,0,31,38000
3,1,35,45000
4,0,52,61000


In [16]:
# Scale age and income
minmax_scaler = MinMaxScaler()
scaled = minmax_scaler.fit_transform(data[['Age', 'income']])
data[['Age', 'income']] = scaled

data


,Gender,Age,income
0,1,0.000000,0.000000
1,0,0.758621,0.750000
2,0,0.275862,0.361111
3,1,0.413793,0.555556
4,0,1.000000,1.000000


## **Summary**

**Scaling (StandardScaler)**: Transforms features to mean=0, std=1 for consistent magnitudes.

**Encoding (LabelEncoder/OneHot)**: Converts categories to numbers (labels) or binary columns (one-hot).

**Normalization (Normalizer):** Scales each row to unit magnitude (length=1) for direction-focused analysis.
